# 이전 문장 시제와 현재 문장 시제의 전환률 및 z점수 계산

- 2.개수_생성함수_sentence_tense.ipynb 파일에서 생성한 개수 생성 csv파일을 바탕으로 작성.


### log 
- ~2026.06.18
    - 입력: "..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_20260520_16-36.csv"
    - 출력: 
    - 필터 적용: df_1101_V = apply_filters(df, {"f_EN_No": 1101, "f_EN_label": "EF", "dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 20, "VX0_No": -1})


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))
from stats.filtering import apply_filters, FilterValue, has_value, _topn_values

from stats.unit_trigram_analysis import (
    analyze_sequence_effect_baseline,
    analyze_unit_sequence_effect,
    analyze_trigram_baseline,
    analyze_unit_trigram_against_baseline,
)
from stats.unit_trigram_plots import (
    plot_effect_scatter,
    plot_binned_summary,
)

## Current analysis module usage

Use `stats.unit_trigram_analysis` for analysis and `stats.unit_trigram_plots` for plots.
Existing ratio and delta columns are kept; smoothed odds-ratio and log-odds-ratio columns are added next to ratio columns.

```python
baseline_tri = analyze_trigram_baseline(df, baseline_col="????")
result = analyze_unit_trigram_against_baseline(
    df,
    baseline_col="????",
    unit_col="V_form",
    baseline_df=baseline_tri,
    smoothing=0.5,
)

ax = plot_effect_scatter(
    result,
    x_col="unit_P_T",
    y_col="switch_return_log_odds_ratio_vs_baseline",
    group_col="????",
    hline=0,
)
```


In [2]:
from datetime import datetime
import pandas as pd
import numpy as np
from pathlib import Path

CSV_PATH = Path(r"..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_20260520_16-36.csv")
        #FILTER_BODY_SEN_ENDS_NOT_QUOTE = lambda df: df[(df['sent_end_V'] == True) & (df['sent_end_V_in_quote'] == False) & (df['speaker'] == 'body')]

COUNT_MODE = "weight" #가중치 파일
WEIGHT_COL="count" #입력 파일이 이미 가중치가 적용된 형태이므로, 가중치 컬럼을 지정하여 "개수" 대신 "가중치 합계"로 계산하도록 함.

df = pd.read_csv(CSV_PATH)

print(f"CSV 파일 로드 완료: {len(df):,}, now: {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

#document 파일 읽기, 'docu_었_결합_등급', 'docu_었_결합_성향' 컬럼만 추출하여 df와 병합
DOCU_CSV = r"..\csv\original_csv\세종문어_document_정보_ver.1.2.csv"
df_docu = pd.read_csv(DOCU_CSV, low_memory=False)
print(f"파일 읽기 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
print(f"df_docu: {df_docu.columns.tolist()}")
df_docu = df_docu[['docu_id', 'category', 'docu_었_결합_등급', 'docu_었_결합_성향']]

print(f"df_docu: {df_docu.columns.tolist()}")
df = df.merge(df_docu, on='docu_id', how='left')
print(f"df_docu와 병합 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

df.loc[df["문서범주"] == "책", "문서범주"] = "인문자연"

df['total'] = True


# 컬럼명 중복 제거(_x, _y)
y_cols = [c for c in df.columns if c.endswith("_y")]
df = df.drop(columns=y_cols)
df.columns = [c[:-2] if c.endswith("_x") else c for c in df.columns]

#문서범주 수정
df.loc[df["문서범주"] == "책", "문서범주"] = "인문자연"

df.columns

CSV 파일 로드 완료: 1,022,399, now: 2026.06.27_15:52:29
파일 읽기 완료 - 2026.06.27_15:52:29
df_docu: ['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4', 'docu_sen_count', 'docu_sen_count_has_E', 'docu_sen_count_not_quote', 'docu_sen_count_has_E_not_quote', 'docu_base_count_not_quote', 'docu_dominant_EN_No', 'docu_dominant_EN_label', 'docu_dominant_count', 'docu_dominant_ratio', 'docu_sent_count', 'docu_head_count', 'docu_body_count', 'docu_body_has_E_count', 'docu_body_not_quote_count', 'docu_body_not_quote_and_었_count', 'docu_었_결합_오즈비', 'docu_었_결합_로그오즈비', 'docu_었_결합_등급', 'docu_었_결합_성향', 'category2', 'file_sent_count', 'file_head_count', 'file_body_count', 'file_body_has_E_count', 'file_body_not_quote_count', 'file_body_not_quote_and_었_count', 'file_었_결합_오즈비', 'file_었_결합_로그오즈비', 'file_었_결합_등급', 'file_었_결합_성향']
df_docu: ['docu_id', 'category', 'docu_었_결합_등급', 'docu_었_결합_성향']
df_docu와 병합 완료 - 2026.06.

Index(['Unnamed: 0', '문서범주', 'category', '매체', 'file_id', 'docu_id', 'speaker',
       'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'V_No',
       'V_form', 'V_label', 'EP_form', 'EN_form', 'EN_label', 'EN_No',
       'EN_No_sub', 'VX_len', 'Next_VX_No', 'Next_VX_form', 'VX0_No',
       'VX0_form', 'VX0_order', 'V0_form', 'V0_No', 'V0_label', 'f_EP_form',
       'f_EN_form', 'f_EN_No', 'f_EN_No_sub', 'f_EN_label',
       'has_prev_sentence', 'has_next_sentence', 'sentence_f_EP_form',
       'prev_sentence_f_EP_form', 'next_sentence_f_EP_form',
       'sentence_sent_end_V_in_quote', 'prev_sentence_sent_end_V_in_quote',
       'next_sentence_sent_end_V_in_quote', 'EP_TT', 'EP_T', 'EP_M', 'f_EP_TT',
       'f_EP_T', 'f_EP_M', 'sentence_f_EP_TT', 'sentence_f_EP_T',
       'sentence_f_EP_M', 'prev_sentence_f_EP_TT', 'prev_sentence_f_EP_T',
 

In [3]:
df.head(5)

,Unnamed: 0,문서범주,category,매체,file_id,docu_id,speaker,sen_count,sen_count_has_E,sen_count_not_quote,...,prev_sentence_f_EP_TT,prev_sentence_f_EP_T,prev_sentence_f_EP_M,next_sentence_f_EP_TT,next_sentence_f_EP_T,next_sentence_f_EP_M,count,docu_었_결합_등급,docu_었_결합_성향,total
0,0,신문,보도해설,신문,AA0001,AA0001.001,body,10,9.0,9.0,...,False,False,False,False,True,False,1,3,중립,True
1,1,신문,보도해설,신문,AA0001,AA0001.001,body,10,9.0,9.0,...,False,True,False,False,False,False,1,3,중립,True
2,2,신문,보도해설,신문,AA0001,AA0001.001,body,10,9.0,9.0,...,False,True,False,False,False,False,1,3,중립,True
3,3,신문,보도해설,신문,AA0001,AA0001.001,body,10,9.0,9.0,...,False,False,False,False,True,False,1,3,중립,True
4,4,신문,보도해설,신문,AA0001,AA0001.001,body,10,9.0,9.0,...,False,False,False,False,True,False,1,3,중립,True


In [22]:
#필터 적용 및 대략적인 데이터 분포 확인``
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))
from stats.filtering import apply_filters, FilterValue, has_value, _topn_values
df_1101 = apply_filters(df, {"f_EN_No": 1101, "f_EN_label": "EF"})
df_V = apply_filters(df, {"VX0_No": -1})
df_docu_selected_V = apply_filters(df, {"dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 20, "VX0_No": -1})
df_docu10_selected_V = apply_filters(df, {"dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 10, "VX0_No": -1})
df_docu10_selected = apply_filters(df, {"dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 10})

df_docu10_EF_V = apply_filters(df, {"dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 10, "VX0_No": -1, "f_EN_label": "EF"})
df_docu10_EF = apply_filters(df, {"dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 10, "f_EN_label": "EF"})


df_신문_selected_V = apply_filters(df_docu_selected_V, {"문서범주": "신문"})


print(f"{len(df_1101):,} rows with f_EN_No=1101, label='EF'(다EF)),\n {len(df_V):,} rows with VX0_No=-1 (V),\n {len(df):,} total rows")
print(df["category"].value_counts(dropna=False))
print(df["prev_sentence_f_EP_T"].value_counts(dropna=False), df["has_prev_sentence"].value_counts(dropna=False))
print(f"{len(df)} rows with unique docu_id: {df['docu_id'].nunique()}")
print(f"{len(df_docu_selected_V)} rows with unique docu_id: {df_docu_selected_V['docu_id'].nunique()}")

834,511 rows with f_EN_No=1101, label='EF'(다EF)),
 716,183 rows with VX0_No=-1 (V),
 1,022,399 total rows
category
인문사회    314801
허구일반    289316
보도해설    171911
체험기술     88817
사설       47616
자연       32611
칼럼       27772
허구아동     27691
총류       21864
Name: count, dtype: int64
prev_sentence_f_EP_T
False    600701
True     421698
Name: count, dtype: int64 has_prev_sentence
True     980438
False     41961
Name: count, dtype: int64
1022399 rows with unique docu_id: 32721
500645 rows with unique docu_id: 9100


In [ ]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")


In [25]:
#Unit 특이성
MOAD = "UNIT_VS_Base"
BASE_COL = 'total'# 'docu_id' #"file_id"#'total'#'문서범주''category'
UNIT_COL = 'V0_No'#'VX_len' #'f_EN_No','VX0_No', 'V0_No' 'VX_len'
TAGET_DF = df_docu10_EF_V

#비교용 기본 df구하기
baseline_df = analyze_trigram_baseline(
    TAGET_DF,
    baseline_col=BASE_COL
)

result = analyze_unit_trigram_against_baseline(
    TAGET_DF,
    baseline_col=BASE_COL,
    unit_col=UNIT_COL,
    baseline_df=baseline_df,
)

# =========================================================
# save to file
# =========================================================
# category 병합
if 'docu_id' in result.columns:
    docu_info = (df[['docu_id', 'category']].drop_duplicates())

    result = result.merge(docu_info, on='docu_id', how='left')
#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"
file_name = SAVE_DIR / f'{MOAD}_by_{UNIT_COL}_{BASE_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now()}***")
result.to_csv(file_name, index=False, encoding="utf-8-sig")
print(f"Output file for pivot table: {file_name}")

c:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\stats\analyze_unit_trigram_ver1.py:1466: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["stay_stay_ratio_vs_baseline"] = _ratio(
c:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\stats\analyze_unit_trigram_ver1.py:1471: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["stay_switch_Obs_rate"] = out["Obs_TTF_rate"] + out["Obs_FFT_rate"]
c:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\stats\analyze_unit_trigram_ver1.py:1472: PerformanceWarning: DataFrame is high

***저장합니다.:    2026-07-01 15:16:08.550294***
Output file for pivot table: ..\csv\결과값\tense\UNIT_VS_Base_by_V0_No_total_2026-07-01_15-13.csv


In [21]:
#연쇄효과
MOAD = "UNIT_sequence_effect"

from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
#UNIT_COL = 'V0_No'#'VX_len' #'f_EN_No','VX0_No', 'V0_No' 'VX_len'

baseline_seq = analyze_sequence_effect_baseline(
    TAGET_DF,
    baseline_col=BASE_COL,
)

result = analyze_unit_sequence_effect(
    TAGET_DF,
    baseline_col=BASE_COL,
    unit_col=UNIT_COL,
    baseline_df=baseline_seq,
    min_unit_n=20,
    min_bigram_n=20,
    min_trigram_n=20,
)


# =========================================================
# 4. save to file
# =========================================================
# category 병합
if 'docu_id' in result.columns:
    docu_info = (df[['docu_id', 'category']].drop_duplicates())

    result = result.merge(docu_info, on='docu_id', how='left')
#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"
file_name = SAVE_DIR / f'{MOAD}_by_{UNIT_COL}_{BASE_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now()}***")
result.to_csv(file_name, index=False, encoding="utf-8-sig")
print(f"Output file for pivot table: {file_name}")

c:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\stats\analyze_unit_trigram_ver1.py:830: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["stay_switch_ratio_vs_expected"] = _ratio(
c:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\stats\analyze_unit_trigram_ver1.py:836: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["return_after_TF_Obs_rate"] = out["P_next_T_given_TF"]
c:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\stats\analyze_unit_trigram_ver1.py:837: PerformanceWarning: DataFrame is highly fragmented.

***저장합니다.:    2026-06-29 22:55:06.391886***
Output file for pivot table: ..\csv\결과값\tense\UNIT_sequence_effect_by_V0_No_category_2026-06-29_22-55.csv


In [ ]:
# 동사, 어미 등 각 unit의 3연쇄를 보는 함수
# 3연쇄 분석에서 unit별 효과를 보는 경우, 2연쇄의 전환률을 baseline과 비교하는 경우, 그리고 3연쇄의 전환/지속률을 baseline과 비교하는 경우를 모두 포함하여 분석하는 코드입니다. 2연쇄의 전환률을 적용하여 3연쇄의 기대값을 계산하는 방식으로, unit의 실제 출현률을 통제한 연쇄 효과를 분석합니다.
# 1. 먼저 P_T를 구하고, baseline의 P_T을 기대값으로 삼아서 차, 확률비를 구한다. 
# 2. 이전문장-현재문장의 2연쇄에서, 이전문장의 T, F비율에 baseline의 T가 주어졌을 때의 T, F확률을 각각 적용한 값을 적용해서 기대값을 구하고 이것과 실제 비율의 차, 확률비를 구한다. 
# 3. 이전문장-현재문장-다음문장의 3연쇄에서, 이전문장 현재문장의 TT, TF, FT, FF가 주어졌을 때의 T, F 확률 값을 각각 적용해서 기대값을 구하고 이것과 실제 비율의 차, 확률비를 구한다. 
# 이걸 구할 수 있도록 baseline의 값을 구하는 함수를 분리해서 코드를 짜고, 이 함수의 결과를 이용해서 실제 내가 구하고 싶은 동사, 어미 등 각 unit의 3연쇄를 보는 함수를 쓴다.


MOAD = "TRI_B"
BASE_COL = 'total'# 'docu_id' #"file_id"#'total'#'문서범주'
UNIT_COL = 'VX0_No' #'f_EN_No'

df_1101 = apply_filters(df, {"f_EN_No": 1101, "f_EN_label": "EF", "dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 20})

from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")


# 1. 전체 baseline
baseline_tri = analyze_trigram_baseline(
    apply_filters(df_1101, {"VX0_No": -1}),
    baseline_col=BASE_COL,
)

# 2. 일부 동사, 어미 등만 분석 3연쇄 분석
verb_tri = analyze_unit_trigram_against_baseline(
    df_1101,
    baseline_col=BASE_COL,
    unit_col=UNIT_COL,
    baseline_df=baseline_tri,
)



# =========================================================
# 4. save to file
# =========================================================
# category 병합
if 'docu_id' in verb_tri.columns:
    docu_info = (df[['docu_id', '문서범주']].drop_duplicates())

    verb_tri = verb_tri.merge(docu_info, on='docu_id', how='left')
#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"
file_name = SAVE_DIR / f'{MOAD}_by_{UNIT_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now()}***")
verb_tri.to_csv(file_name, index=False, encoding="utf-8-sig")
print(f"Output file for pivot table: {file_name}")

***저장합니다.:    2026-05-29 16:21:01.045244***
Output file for pivot table: ..\csv\결과값\tense\TRI_B_by_VX0_No_2026-05-29_16-20.csv
